# Analysis: semi-supervised learning for rare CWE types

This notebook runs the full analysis for the paper. Run the cells from top to bottom. All design choices are in `decisions.md` in the project root.

## 1. Libraries

This cell loads all the libraries we need. We use pandas and numpy for the data, PyTorch and Hugging Face `transformers` for the CodeBERT model, scikit-learn for the F1 scores, and matplotlib for the plots. The Hugging Face `Trainer` also needs the package `accelerate` to be installed.

In [ ]:
from pathlib import Path
import json
import random
import shutil

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from huggingface_hub import snapshot_download
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

## 2. Setup

This cell sets where the three datasets are and the output folders `results/` and `figures/`. It creates these folders if they do not exist yet. The folder `checkpoints/` is only used for a short time during training (see section 5).

DiverseVul comes from the Hugging Face Hub (`claudios/DiverseVul`), so you do not need a local copy. We fix the dataset version (`DIVERSEVUL_REVISION`), so the data stays the same even if the dataset changes later. BigVul and MegaVul are local files in `Data/`.

The function `set_seed` fixes the random numbers in Python, numpy and PyTorch. With the same seed, a run gives the same result again.

We do not choose the hardware (GPU or CPU) ourselves. The Hugging Face `Trainer` does this for us: it uses a CUDA GPU if there is one, then an Apple GPU (MPS), and otherwise the CPU.

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DATA_DIR = ROOT / "Data"
DIVERSEVUL_REPO = "claudios/DiverseVul"
DIVERSEVUL_REVISION = "98c294b3caaa94170e77282f1587ea20c3d59784"
BIGVUL_PATH = DATA_DIR / "BigVul" / "MSR_data_cleaned.csv"
MEGAVUL_PATH = DATA_DIR / "MegaVul" / "megavul_simple.json"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"
CHECKPOINT_DIR = ROOT / "checkpoints"
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


print("BigVul:", BIGVUL_PATH.exists(), "| MegaVul:", MEGAVUL_PATH.exists())

## 3. Load and clean DiverseVul

This cell downloads DiverseVul from the Hugging Face Hub (about 170 MB). It does this only the first time: after that, it uses the saved copy in the Hugging Face cache. The data is the same as the original file `diversevul_20230702.json`, with the same rows in the same order. We checked this row by row.

The cell builds one clean table, called `data`, with one row per function.

- We keep only functions that are vulnerable, have at least one CWE label, and have code.
- Some functions have more than one CWE. We give each function the rarest CWE in its list, because rare classes are the focus of our study. If two CWEs are equally rare, we take the first one in the list.
- We drop CWE classes with only 1 function (`MIN_COUNT = 2`).

The result has 16,098 functions, 139 CWE classes and 739 projects. We keep the columns `func` (the code), `cwe_label` (the class), `project` and `commit_id`. The model only uses `func`. We need `project` for the split in the next cell.

In [ ]:
MIN_COUNT = 2

diversevul_dir = Path(snapshot_download(DIVERSEVUL_REPO, repo_type="dataset", revision=DIVERSEVUL_REVISION))
diversevul = pd.concat(pd.read_parquet(path) for path in sorted(diversevul_dir.glob("data/*.parquet"))).reset_index(drop=True)
diversevul["cwe"] = diversevul["cwe"].map(list)

labeled = diversevul[
    (diversevul["target"] == 1)
    & (diversevul["cwe"].map(len) > 0)
    & (diversevul["func"].astype(str).str.strip() != "")
].copy()

cwe_freq = labeled["cwe"].explode().value_counts()


# We take the rarest CWE when multiple exists
def pick_cwe_label(cwe_list):
    return min(cwe_list, key=lambda cwe: (cwe_freq[cwe], cwe_list.index(cwe)))


labeled["cwe_label"] = labeled["cwe"].map(pick_cwe_label)
label_counts = labeled["cwe_label"].value_counts()
labeled = labeled[labeled["cwe_label"].isin(label_counts[label_counts >= MIN_COUNT].index)]

data = labeled[["func", "cwe_label", "project", "commit_id"]].reset_index(drop=True)

print("Raw rows:", len(diversevul))
print("Rows:", len(data))
print("Classes:", data["cwe_label"].nunique())
print("Projects:", data["project"].nunique())

## 4. Train, validation and test split

This cell splits the data by project. All functions from one project go to only one part: train, validation (val) or test. This stops the model from learning code from a project and then seeing the same project again in the test.

We shuffle the projects with seed 1060 and put 70% of them in train, 15% in val and 15% in test. We chose this seed because it gives the most rare classes in the test set, while each part keeps close to 70/15/15 of the rows. The paper must say this. The split is saved in `results/project_split.csv`.

We keep only classes with at least 10 functions in train (`MIN_TRAIN = 10`). We remove all other functions from train, val and test. This includes val and test functions whose class does not appear in train at all, because the model can never predict these classes. We made this choice because classes with fewer than 10 train functions were almost impossible to learn: in our first full run with all 129 classes, the rare-class F1 was close to 0, even with all labels. The removed classes are 47 of the 129 train classes, but only 212 train functions (2%). In total, we remove 212 functions from train, 88 from val and 201 from test.

A class is rare when it has fewer than 50 functions in train (`RARE_MAX = 50`). Otherwise it is common. So a rare class has 10 to 49 train functions. The largest class (CWE-787) has 1,140.

After this, train has 82 classes (47 rare, 35 common). Test has 65 classes, and 33 of them are rare, with 306 test functions in total.

The split seed was chosen before this filter, with the old rare definition (fewer than 20 train functions). We kept the same seed, so the projects in train, val and test do not change.

In [ ]:
SPLIT_SEED = 1060
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
MIN_TRAIN = 10
RARE_MAX = 50

projects = data["project"].drop_duplicates().sort_values().to_numpy()
order = np.random.RandomState(SPLIT_SEED).permutation(len(projects))
n_train = int(TRAIN_FRAC * len(projects))
n_val = int(VAL_FRAC * len(projects))

split_of_project = pd.Series("test", index=projects)
split_of_project.iloc[order[:n_train]] = "train"
split_of_project.iloc[order[n_train:n_train + n_val]] = "val"
data["split"] = data["project"].map(split_of_project)
split_of_project.rename_axis("project").rename("split").to_csv(RESULTS_DIR / "project_split.csv")

train_df = data[data["split"] == "train"].reset_index(drop=True)
val_df = data[data["split"] == "val"].reset_index(drop=True)
test_df = data[data["split"] == "test"].reset_index(drop=True)

all_train_counts = train_df["cwe_label"].value_counts()
kept_classes = all_train_counts[all_train_counts >= MIN_TRAIN].index
print("Classes in train:", len(all_train_counts), "| kept (at least", MIN_TRAIN, "train functions):", len(kept_classes))
print("Rows dropped (class not kept) - train:", int((~train_df["cwe_label"].isin(kept_classes)).sum()), "| val:", int((~val_df["cwe_label"].isin(kept_classes)).sum()), "| test:", int((~test_df["cwe_label"].isin(kept_classes)).sum()))
train_df = train_df[train_df["cwe_label"].isin(kept_classes)].reset_index(drop=True)
val_df = val_df[val_df["cwe_label"].isin(kept_classes)].reset_index(drop=True)
test_df = test_df[test_df["cwe_label"].isin(kept_classes)].reset_index(drop=True)

train_counts = train_df["cwe_label"].value_counts()
rare_classes = set(train_counts[train_counts < RARE_MAX].index)
test_classes = set(test_df["cwe_label"])

splits = {"train": train_df, "val": val_df, "test": test_df}
print(pd.DataFrame({name: {"projects": df["project"].nunique(), "rows": len(df), "classes": df["cwe_label"].nunique()} for name, df in splits.items()}).T)
print("Rare classes in train:", len(rare_classes), "| Common classes in train:", len(train_counts) - len(rare_classes))
print("Test classes:", len(test_classes), "| of which rare:", len(test_classes & rare_classes), "| rare test rows:", int(test_df["cwe_label"].isin(rare_classes).sum()))

## 5. Training code

This cell holds the code to train and use the model. It does not train anything yet. We use the Hugging Face `Trainer`, which contains the training loop, so we do not have to write it ourselves.

- **Model:** `microsoft/codebert-base` with a classification head. The head is a small extra layer that picks one of the 82 CWE classes in train.
- **Input:** the model reads at most 512 tokens of each function. A token is a small piece of code, such as a word or a symbol. Longer functions are cut off at the end.
- **Loss:** class-balanced loss. Mistakes on rare classes count more, so the model does not only learn the common classes. The normal `Trainer` does not support class weights, so we make a small subclass, `WeightedTrainer`, that changes only the loss.
- **Training:** batch size 16, learning rate 2e-5, 10 epochs. An epoch is one full pass over the training data. In the first 10% of the training steps, the learning rate slowly goes up from 0 to 2e-5 (warmup). This makes the start of training more stable. After each epoch, the `Trainer` computes macro-F1 on val and saves the model in `checkpoints/`. At the end, it loads the epoch with the best val macro-F1. Then we delete `checkpoints/`, because the files are large (about 0.5 GB each).
- **Why these settings:** with all 129 classes, our first settings (learning rate 2e-5, 5 epochs, no warmup) gave a val macro-F1 of only 0.029. In a quick check on val, a learning rate of 5e-5 with warmup and 10 epochs gave 0.065. The check also showed that the class weights help: without them, the rare-class F1 was 0. But in our first full run with 5e-5, 3 of the 30 models on small labeled sets did not learn at all: the train loss stayed at the level of random guessing for (almost) all 10 epochs. So we went back to 2e-5 and kept the warmup and the 10 epochs. See `decisions.md`.

Functions:

- `train_model(texts, labels, seed, epochs)` trains a new model. It trains for 10 epochs, unless you give another number of epochs (we only do this in section 11). It returns the trainer (which holds the model) and the training history.
- `predict_proba(trainer, texts)` gives the probability of each class for each function. We will also use it for pseudo-labeling later.
- `predict_labels(trainer, texts)` gives the most likely class for each function.
- `macro_f1(y_true, y_pred)` computes macro-F1. Macro-F1 is the mean F1 over all classes, so every class counts the same.
- `per_class_f1(y_true, y_pred)` gives a table with the F1 of each class, the number of train functions of that class, and its group (rare or common).

In [ ]:
MODEL_NAME = "microsoft/codebert-base"
MAX_LENGTH = 512
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WARMUP = 0.1
EPOCHS = 10

classes = sorted(train_counts.index)
label2id = {label: i for i, label in enumerate(classes)}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)


class CodeDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH)
        self.label_ids = None if labels is None else [label2id[label] for label in labels]

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, i):
        item = {key: values[i] for key, values in self.encodings.items()}
        if self.label_ids is not None:
            item["labels"] = self.label_ids[i]
        return item


class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=sorted(set(y_true)), average="macro", zero_division=0)


def per_class_f1(y_true, y_pred):
    labels = sorted(set(y_true))
    result = pd.DataFrame({
        "cwe": labels,
        "f1": f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0),
        "train_count": train_counts[labels].to_numpy(),
    })
    result["group"] = np.where(result["cwe"].isin(rare_classes), "rare", "common")
    return result


def compute_metrics(pred):
    y_true = np.array(classes)[pred.label_ids]
    y_pred = np.array(classes)[pred.predictions.argmax(axis=1)]
    return {"macro_f1": macro_f1(y_true, y_pred)}


def train_model(texts, labels, seed, epochs=EPOCHS):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(classes))
    train_data = CodeDataset(texts, labels)
    weights = compute_class_weight("balanced", classes=np.arange(len(classes)), y=train_data.label_ids)

    args = TrainingArguments(
        output_dir=CHECKPOINT_DIR,
        num_train_epochs=epochs,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        save_total_limit=1,
        save_only_model=True,
        report_to="none",
        seed=seed,
    )
    trainer = WeightedTrainer(
        class_weights=torch.tensor(weights, dtype=torch.float32),
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=CodeDataset(val_df["func"], val_df["cwe_label"]),
        data_collator=collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)

    history = pd.DataFrame(trainer.state.log_history).groupby("epoch").first().reset_index()
    return trainer, history


def predict_proba(trainer, texts):
    logits = trainer.predict(CodeDataset(texts)).predictions
    return torch.softmax(torch.tensor(logits), dim=1).numpy()


def predict_labels(trainer, texts):
    return np.array(classes)[predict_proba(trainer, texts).argmax(axis=1)]

## 6. Reference model with all labels

This cell trains one model on all labels in train (100%). This model has two goals. First, it checks that the training code works. Second, it shows the best result we can expect. Later we compare the models with few labels to this result.

The cell prints the test macro-F1 and the mean F1 of the rare and common classes. It saves the training history and the F1 of each class in `results/`. When these files already exist, the cell does not train again. It only reads the saved results.

This cell is slow. On an RTX 4090 GPU it takes about 30 minutes. On a MacBook (MPS), one epoch takes about 35 minutes, so the 10 epochs take about 6 hours.

In [ ]:
FULL_SEED = 0
FULL_PATH = RESULTS_DIR / "full_supervised_per_class.csv"

if FULL_PATH.exists():
    print("Reference model already done")
    full_per_class = pd.read_csv(FULL_PATH)
else:
    full_trainer, full_history = train_model(train_df["func"], train_df["cwe_label"], FULL_SEED)
    full_test_pred = predict_labels(full_trainer, test_df["func"])
    full_per_class = per_class_f1(test_df["cwe_label"], full_test_pred)
    full_history.to_csv(RESULTS_DIR / "full_supervised_history.csv", index=False)
    full_per_class.to_csv(FULL_PATH, index=False)

print("Test macro-F1:", round(full_per_class["f1"].mean(), 4))
print(full_per_class.groupby("group")["f1"].agg(["mean", "count"]))

## 7. Labeled set and unlabeled pool

In this study, labels are scarce on purpose. This cell builds the small labeled set and the large unlabeled pool from the train split. Val and test do not change: they keep all their labels.

- **Label sizes:** we keep the labels of 5%, 10% or 25% of the functions in each class (`LABEL_FRACS`).
- **At least 1 label per class:** every class keeps at least 1 labeled function, also when 5% of the class is less than 1. So every class is in the labeled set. We round in the normal way: 1.5 becomes 2.
- **Seeds:** for each label size we use 5 seeds (`SEEDS`). Each seed chooses other functions for the labeled set. This shows how much the results depend on which functions get a label.
- **Unlabeled pool:** all other train functions. Their true class is kept in the column `hidden_label`. The models never use this column. We only use it later to check how many pseudo-labels are correct.

The function `make_label_split(frac, seed)` returns the labeled set and the unlabeled pool for one label size and one seed. The table below shows the sizes. The sizes are the same for every seed, because only the chosen functions change, not how many.

In [ ]:
LABEL_FRACS = [0.05, 0.10, 0.25]
SEEDS = [0, 1, 2, 3, 4]


def make_label_split(frac, seed):
    shuffled = train_df.sample(frac=1, random_state=seed)
    n_labeled = shuffled["cwe_label"].map(lambda label: max(1, int(frac * train_counts[label] + 0.5)))
    is_labeled = shuffled.groupby("cwe_label").cumcount() < n_labeled
    labeled_df = shuffled[is_labeled].reset_index(drop=True)
    unlabeled_df = shuffled[~is_labeled].rename(columns={"cwe_label": "hidden_label"}).reset_index(drop=True)
    return labeled_df, unlabeled_df


label_summary = []
for frac in LABEL_FRACS:
    labeled_df, unlabeled_df = make_label_split(frac, SEEDS[0])
    labels_per_class = labeled_df["cwe_label"].value_counts()
    label_summary.append({
        "label_frac": frac,
        "labeled_rows": len(labeled_df),
        "unlabeled_rows": len(unlabeled_df),
        "classes": labels_per_class.size,
        "classes_with_1_label": int((labels_per_class == 1).sum()),
        "rare_classes_with_1_label": int((labels_per_class[list(rare_classes)] == 1).sum()),
    })

print(pd.DataFrame(label_summary).to_string(index=False))

## 8. Supervised baseline

This is the baseline for RQ1. For each label size (5%, 10%, 25%) and each seed (0 to 4), we train a model only on the small labeled set from section 7. That gives 15 models. The SSL model in the next section gets exactly the same labeled set, plus the unlabeled pool. So the only difference between the two models is the unlabeled data.

For each run, the cell:

- trains a model on the labeled set with `train_model`;
- predicts the test set and saves the F1 of each class;
- predicts the unlabeled pool and saves the probabilities. The SSL step uses these probabilities to make pseudo-labels, so we do not have to train this model again;
- saves the model in `models/baseline/`, so we can test it on BigVul and MegaVul later (RQ3). Each model is about 0.5 GB, so all 15 together are about 7.5 GB.

All files go to `results/baseline/`, with names such as `frac0.05_seed0_per_class.csv`. When a run is already done, the cell skips it. So you can stop the notebook and start it again later, and it goes on with the next run.

This cell is slow. On an RTX 4090 GPU, all 15 runs take about 3 to 5 hours in total. On a MacBook (MPS), they take about 20 hours or more.

In [ ]:
BASELINE_DIR = RESULTS_DIR / "baseline"
MODEL_DIR = ROOT / "models"
BASELINE_DIR.mkdir(exist_ok=True)

for frac in LABEL_FRACS:
    for seed in SEEDS:
        name = f"frac{frac}_seed{seed}"
        if (BASELINE_DIR / f"{name}_per_class.csv").exists():
            print(name, "already done")
            continue

        labeled_df, unlabeled_df = make_label_split(frac, seed)
        trainer, history = train_model(labeled_df["func"], labeled_df["cwe_label"], seed)
        test_pred = predict_labels(trainer, test_df["func"])

        np.save(BASELINE_DIR / f"{name}_unlabeled_proba.npy", predict_proba(trainer, unlabeled_df["func"]))
        trainer.save_model(MODEL_DIR / "baseline" / name)
        history.to_csv(BASELINE_DIR / f"{name}_history.csv", index=False)
        per_class_f1(test_df["cwe_label"], test_pred).to_csv(BASELINE_DIR / f"{name}_per_class.csv", index=False)
        print(name, "test macro-F1:", round(macro_f1(test_df["cwe_label"], test_pred), 4))

### Baseline results

This cell reads the saved results of all 15 runs. For each label size, it shows the mean and the standard deviation over the 5 seeds of three numbers: the macro-F1 over all test classes, the mean F1 of the rare classes, and the mean F1 of the common classes. The table is saved in `results/baseline_summary.csv`.

In [ ]:
baseline = pd.concat(
    pd.read_csv(BASELINE_DIR / f"frac{frac}_seed{seed}_per_class.csv").assign(label_frac=frac, seed=seed)
    for frac in LABEL_FRACS
    for seed in SEEDS
)

macro = baseline.groupby(["label_frac", "seed"])["f1"].mean().rename("macro_f1")
groups = baseline.groupby(["label_frac", "seed", "group"])["f1"].mean().unstack("group")
baseline_summary = pd.concat([macro, groups], axis=1).groupby("label_frac").agg(["mean", "std"]).round(3)

baseline_summary.to_csv(RESULTS_DIR / "baseline_summary.csv")
print(baseline_summary)

## 9. Semi-supervised learning (pseudo-labeling)

This is the SSL model for RQ1 and RQ2. It uses plain pseudo-labeling, in one round:

1. We take the predictions of the baseline model (section 8) on the unlabeled pool. We saved these as probabilities, so we do not have to train the baseline again.
2. For each function in the unlabeled pool, the confidence of the model is its highest class probability. We take the 20% of the pool with the highest confidence (`PSEUDO_FRAC = 0.2`) and use the predicted class as their label. Such a label is called a pseudo-label. We use one rule for all classes, and we fix it in advance instead of tuning it, to keep the design simple. We do not use a fixed probability threshold, such as 0.5: our models are not confident (with 82 classes, the highest probability is often below 0.1), so in our first full run a threshold of 0.5 gave no pseudo-labels at all in 10 of the 15 runs. Taking the top 20% always gives pseudo-labels, however confident the model is.
3. We train a new model on the labeled set plus the pseudo-labeled functions. Pseudo-labeled functions count the same as labeled functions in the loss. All other settings are the same as for the baseline (section 5), including the seed. So the only difference with the baseline is the extra pseudo-labeled data.

The unlabeled pool still holds the true class of each function in the column `hidden_label`. The models never use it. We use it only to measure how many pseudo-labels are correct. We report this separately for pseudo-labels of rare classes and of common classes, because it shows *why* SSL helps a group or not.

The function `make_pseudo_set(frac, seed)` gives the labeled set and the pseudo-labeled functions for one run. The function `pseudo_stats(pseudo_df)` counts the pseudo-labels and the share that is correct (precision), for all, rare and common classes. A rare or common pseudo-label means: the model gave it a rare or a common class.

In [ ]:
PSEUDO_FRAC = 0.2
SSL_DIR = RESULTS_DIR / "ssl"
SSL_DIR.mkdir(exist_ok=True)


def make_pseudo_set(frac, seed):
    labeled_df, unlabeled_df = make_label_split(frac, seed)
    proba = np.load(BASELINE_DIR / f"frac{frac}_seed{seed}_unlabeled_proba.npy")
    assert len(proba) == len(unlabeled_df)
    pseudo_df = unlabeled_df.assign(cwe_label=np.array(classes)[proba.argmax(axis=1)], confidence=proba.max(axis=1))
    pseudo_df = pseudo_df.nlargest(int(PSEUDO_FRAC * len(pseudo_df)), "confidence").reset_index(drop=True)
    return labeled_df, pseudo_df


def pseudo_stats(pseudo_df):
    correct = pseudo_df["cwe_label"] == pseudo_df["hidden_label"]
    is_rare = pseudo_df["cwe_label"].isin(rare_classes)
    return {
        "pseudo_rows": len(pseudo_df),
        "pseudo_precision": correct.mean(),
        "pseudo_rows_rare": int(is_rare.sum()),
        "pseudo_precision_rare": correct[is_rare].mean(),
        "pseudo_rows_common": int((~is_rare).sum()),
        "pseudo_precision_common": correct[~is_rare].mean(),
    }

### SSL runs

For each label size and seed, this cell trains one SSL model. It uses the same 15 labeled sets as the baseline. For each run, it saves:

- the F1 of each test class, in `results/ssl/`, with the same file names as the baseline;
- the training history;
- the pseudo-label counts and precision (`*_pseudo.csv`);
- the model, in `models/ssl/`, so we can test it on BigVul and MegaVul later (RQ3). All 15 models together are about 7.5 GB.

When a run is already done, the cell skips it.

This cell is slow. Each SSL model trains on more functions than the baseline (labeled plus pseudo-labeled), so all 15 runs take about 4 to 6 hours on an RTX 4090.

In [ ]:
for frac in LABEL_FRACS:
    for seed in SEEDS:
        name = f"frac{frac}_seed{seed}"
        if (SSL_DIR / f"{name}_per_class.csv").exists():
            print(name, "already done")
            continue

        labeled_df, pseudo_df = make_pseudo_set(frac, seed)
        train_set = pd.concat([labeled_df, pseudo_df])
        trainer, history = train_model(train_set["func"], train_set["cwe_label"], seed)
        test_pred = predict_labels(trainer, test_df["func"])

        trainer.save_model(MODEL_DIR / "ssl" / name)
        history.to_csv(SSL_DIR / f"{name}_history.csv", index=False)
        pd.DataFrame([pseudo_stats(pseudo_df)]).to_csv(SSL_DIR / f"{name}_pseudo.csv", index=False)
        per_class_f1(test_df["cwe_label"], test_pred).to_csv(SSL_DIR / f"{name}_per_class.csv", index=False)
        print(name, "pseudo-labels:", len(pseudo_df), "| test macro-F1:", round(macro_f1(test_df["cwe_label"], test_pred), 4))

### SSL results (RQ1)

This cell compares SSL with the baseline. For each label size, it shows the mean and the standard deviation over the 5 seeds of the macro-F1, the mean F1 of the rare classes, and the mean F1 of the common classes, for the baseline, for SSL, and for the gain (SSL minus baseline). A positive gain means that SSL is better.

The second table shows the pseudo-labels: how many there are and how many are correct, for rare and common classes, as a mean over the 5 seeds.

The tables are saved in `results/ssl_summary.csv` and `results/ssl_pseudo_summary.csv`.

In [ ]:
def load_runs(folder):
    return pd.concat(
        pd.read_csv(folder / f"frac{frac}_seed{seed}_per_class.csv").assign(label_frac=frac, seed=seed)
        for frac in LABEL_FRACS
        for seed in SEEDS
    )


runs = pd.concat([load_runs(BASELINE_DIR).assign(model="baseline"), load_runs(SSL_DIR).assign(model="ssl")])
macro = runs.groupby(["label_frac", "seed", "model"])["f1"].mean().rename("macro_f1")
groups = runs.groupby(["label_frac", "seed", "model", "group"])["f1"].mean().unstack("group")
per_run = pd.concat([macro, groups], axis=1).unstack("model")
for metric in ["macro_f1", "rare", "common"]:
    per_run[(metric, "gain")] = per_run[(metric, "ssl")] - per_run[(metric, "baseline")]
ssl_summary = per_run.sort_index(axis=1).groupby("label_frac").agg(["mean", "std"]).round(3)
ssl_summary.to_csv(RESULTS_DIR / "ssl_summary.csv")
print(ssl_summary.to_string())

pseudo = pd.concat(
    pd.read_csv(SSL_DIR / f"frac{frac}_seed{seed}_pseudo.csv").assign(label_frac=frac, seed=seed)
    for frac in LABEL_FRACS
    for seed in SEEDS
)
pseudo_summary = pseudo.drop(columns="seed").groupby("label_frac").mean().round(3)
pseudo_summary.to_csv(RESULTS_DIR / "ssl_pseudo_summary.csv")
print()
print(pseudo_summary.to_string())

### Gain per class (RQ2)

This cell answers RQ2: is the SSL gain larger for rare classes than for common classes? For each test class and label size, it computes the F1 gain (SSL minus baseline), as a mean over the 5 seeds.

- The figure shows the gain of each class against the number of train functions in that class. The x-axis uses a log scale. The dashed line is the border between rare and common (`RARE_MAX`). The figure is saved in `figures/ssl_gain_per_class.pdf`.
- The gain of each class is also saved in `results/ssl_gain_per_class.csv`. The mean gain of the rare and common groups is in the RQ1 table above.

In [ ]:
per_class_gain = runs.groupby(["label_frac", "cwe", "train_count", "group", "model"])["f1"].mean().unstack("model").reset_index()
per_class_gain["gain"] = per_class_gain["ssl"] - per_class_gain["baseline"]
per_class_gain.to_csv(RESULTS_DIR / "ssl_gain_per_class.csv", index=False)

fig, axes = plt.subplots(1, len(LABEL_FRACS), figsize=(12, 3.5), sharey=True)
for ax, frac in zip(axes, LABEL_FRACS):
    part = per_class_gain[per_class_gain["label_frac"] == frac]
    for group, color in [("rare", "tab:orange"), ("common", "tab:blue")]:
        points = part[part["group"] == group]
        ax.scatter(points["train_count"], points["gain"], s=14, color=color, label=group)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.axvline(RARE_MAX, color="grey", linewidth=0.8, linestyle="--")
    ax.set_xscale("log")
    ax.set_title(f"{frac:.0%} labels")
    ax.set_xlabel("Train functions in class")
axes[0].set_ylabel("F1 gain (SSL - baseline)")
axes[0].legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "ssl_gain_per_class.pdf")
plt.show()

## 10. Generalization to BigVul and MegaVul (RQ3)

RQ3 asks if the result also holds outside DiverseVul. We test the saved baseline and SSL models from sections 8 and 9 on two other datasets: BigVul and MegaVul. We do not train or tune anything on these datasets.

### Clean BigVul and MegaVul

This cell builds one table with the test functions of both datasets. It uses the same rules as for DiverseVul:

- We keep only vulnerable functions with a CWE label and with code. The code is the vulnerable version of the function (`func_before`).
- We remove labels that are not a real CWE number, such as `CWE-Other` and `NVD-CWE-noinfo`.
- Each function gets one label: the rarest CWE in its list, counted within that dataset. This is the same rule as in section 3.
- We remove duplicate functions. Two functions are the same when their code is the same after we remove all spaces, tabs and new lines.
- We remove every function that is also a vulnerable function in DiverseVul (compared in the same way). Many functions are in more than one dataset, because the datasets collect fixes from the same projects. Without this step, we would test the models on functions they may have seen in training.
- We keep only functions whose class is one of the 82 classes in DiverseVul train, because the model can only predict these classes. The model still predicts over all 82 classes, so a wrong class counts as an error. Rare and common use the same groups as before, based on the DiverseVul train counts.

The raw BigVul file is large (10 GB), so we run this cell once on a laptop and save the result in `results/rq3_targets.csv`. When this file exists, the cell only reads it. So you only need to upload this small file to the GPU machine, not the raw datasets.

Result: BigVul has 4,975 functions in 55 classes (24 rare, with 297 functions). MegaVul has 10,394 functions in 77 classes (44 rare, with 1,275 functions). The overlap step removes many functions: 3,673 from BigVul and 5,195 from MegaVul.

In [ ]:
RQ3_PATH = RESULTS_DIR / "rq3_targets.csv"


def no_whitespace(code):
    return "".join(str(code).split())


def clean_target(df):
    df = df.copy()
    df["cwe"] = df["cwe"].map(lambda cwes: [cwe for cwe in cwes if cwe.startswith("CWE-") and cwe[4:].isdigit()])
    df = df[(df["cwe"].map(len) > 0) & (df["func"].astype(str).str.strip() != "")]
    freq = df["cwe"].explode().value_counts()
    df["cwe_label"] = df["cwe"].map(lambda cwes: min(cwes, key=lambda cwe: (freq[cwe], cwes.index(cwe))))
    df["code_key"] = df["func"].map(no_whitespace)
    n_rows = len(df)
    df = df.drop_duplicates("code_key")
    n_unique = len(df)
    df = df[~df["code_key"].isin(diversevul_code)]
    print("Rows:", n_rows, "| after removing duplicates:", n_unique, "| after removing DiverseVul overlap:", len(df))
    return df[df["cwe_label"].isin(train_counts.index)][["func", "cwe_label", "project"]]


if RQ3_PATH.exists():
    print("RQ3 data already done")
    targets = pd.read_csv(RQ3_PATH)
else:
    diversevul_code = set(diversevul.loc[diversevul["target"] == 1, "func"].map(no_whitespace))

    bigvul = pd.read_csv(BIGVUL_PATH, usecols=["CWE ID", "func_before", "vul", "project"], dtype={"CWE ID": str})
    bigvul = bigvul[bigvul["vul"] == 1]
    bigvul = pd.DataFrame({
        "func": bigvul["func_before"],
        "cwe": bigvul["CWE ID"].map(lambda cwe: [cwe] if isinstance(cwe, str) else []),
        "project": bigvul["project"],
    })

    megavul = pd.DataFrame(json.load(open(MEGAVUL_PATH)))
    megavul = megavul[megavul["is_vul"] == True]
    megavul = pd.DataFrame({"func": megavul["func_before"], "cwe": megavul["cwe_ids"], "project": megavul["repo_name"]})

    targets = pd.concat([clean_target(bigvul).assign(dataset="bigvul"), clean_target(megavul).assign(dataset="megavul")], ignore_index=True)
    targets.to_csv(RQ3_PATH, index=False)

targets["group"] = np.where(targets["cwe_label"].isin(rare_classes), "rare", "common")
print(targets.groupby(["dataset", "group"]).agg(rows=("func", "size"), classes=("cwe_label", "nunique")))

### Test the saved models

This cell loads each saved model (15 baseline and 15 SSL models) and predicts the class of every function in BigVul and MegaVul. For each model and dataset, it saves the F1 of each class in `results/rq3/`, with names such as `bigvul_ssl_frac0.05_seed0_per_class.csv`. When a file already exists, the cell skips that model. The reference model from section 6 is not included, because we did not save it.

This cell needs the folder `models/` from sections 8 and 9. On an RTX 4090, it takes about 30 minutes.

In [ ]:
RQ3_DIR = RESULTS_DIR / "rq3"
RQ3_DIR.mkdir(exist_ok=True)
DATASETS = ["bigvul", "megavul"]

target_data = {dataset: CodeDataset(targets.loc[targets["dataset"] == dataset, "func"]) for dataset in DATASETS}
predict_args = TrainingArguments(output_dir=CHECKPOINT_DIR, per_device_eval_batch_size=BATCH_SIZE * 2, report_to="none")

for model_type in ["baseline", "ssl"]:
    for frac in LABEL_FRACS:
        for seed in SEEDS:
            name = f"frac{frac}_seed{seed}"
            if all((RQ3_DIR / f"{dataset}_{model_type}_{name}_per_class.csv").exists() for dataset in DATASETS):
                print(model_type, name, "already done")
                continue

            model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR / model_type / name)
            trainer = Trainer(model=model, args=predict_args, data_collator=collator)
            for dataset in DATASETS:
                y_true = targets.loc[targets["dataset"] == dataset, "cwe_label"]
                y_pred = np.array(classes)[trainer.predict(target_data[dataset]).predictions.argmax(axis=1)]
                per_class_f1(y_true, y_pred).to_csv(RQ3_DIR / f"{dataset}_{model_type}_{name}_per_class.csv", index=False)
                print(model_type, name, dataset, "macro-F1:", round(macro_f1(y_true, y_pred), 4))

### RQ3 results

This cell makes the same table as for RQ1, but for BigVul and MegaVul. For each dataset and label size, it shows the mean and the standard deviation over the 5 seeds of the macro-F1, the mean F1 of the rare classes, and the mean F1 of the common classes, for the baseline, for SSL, and for the gain (SSL minus baseline). The table is saved in `results/rq3_summary.csv`.

In [ ]:
rq3_runs = pd.concat(
    pd.read_csv(RQ3_DIR / f"{dataset}_{model_type}_frac{frac}_seed{seed}_per_class.csv").assign(dataset=dataset, model=model_type, label_frac=frac, seed=seed)
    for dataset in DATASETS
    for model_type in ["baseline", "ssl"]
    for frac in LABEL_FRACS
    for seed in SEEDS
)
keys = ["dataset", "label_frac", "seed", "model"]
macro = rq3_runs.groupby(keys)["f1"].mean().rename("macro_f1")
groups = rq3_runs.groupby(keys + ["group"])["f1"].mean().unstack("group")
rq3_per_run = pd.concat([macro, groups], axis=1).unstack("model")
for metric in ["macro_f1", "rare", "common"]:
    rq3_per_run[(metric, "gain")] = rq3_per_run[(metric, "ssl")] - rq3_per_run[(metric, "baseline")]
rq3_summary = rq3_per_run.sort_index(axis=1).groupby(["dataset", "label_frac"]).agg(["mean", "std"]).round(3)
rq3_summary.to_csv(RESULTS_DIR / "rq3_summary.csv")
print(rq3_summary.to_string())

## 11. Extra control: baseline with the same number of training steps

The SSL model trains on the labeled set plus the pseudo-labels, for 10 epochs. So it makes more training steps than the baseline: about 4.8 times more at 5% labels, 2.8 times more at 10% and 1.6 times more at 25%. Part of the SSL gain may come from this longer training, and not from the unlabeled data. In the full run, the small baselines were still improving at the end of training, and SSL helped a little at 5% labels although 94% of the pseudo-labels were wrong.

To check this, we train the baseline again, on the same labeled set, but with as many training steps as the SSL model. We call it the long baseline. Its number of epochs is 10 × (rows in the SSL training set / rows in the labeled set), rounded: 48, 28 and 16 epochs for 5%, 10% and 25% labels. All other settings are the same as in section 5, including the seed. We still keep the epoch with the best val macro-F1.

This is an extra control. We added it after we saw the test results. The main baseline (section 8) does not change.

For each run, the cell saves the F1 of each test class and the training history in `results/long_baseline/`. It also tests the model on BigVul and MegaVul and saves the results in `results/rq3/`, with names such as `bigvul_long_baseline_frac0.05_seed0_per_class.csv`. We do not save the models. When a run is already done, the cell skips it.

This cell is slow. All 15 runs take about 4 to 6 hours on an RTX 4090.

In [ ]:
LONG_DIR = RESULTS_DIR / "long_baseline"
LONG_DIR.mkdir(exist_ok=True)

for frac in LABEL_FRACS:
    for seed in SEEDS:
        name = f"frac{frac}_seed{seed}"
        if (LONG_DIR / f"{name}_per_class.csv").exists():
            print(name, "already done")
            continue

        labeled_df, pseudo_df = make_pseudo_set(frac, seed)
        epochs = round(EPOCHS * (len(labeled_df) + len(pseudo_df)) / len(labeled_df))
        trainer, history = train_model(labeled_df["func"], labeled_df["cwe_label"], seed, epochs=epochs)
        test_pred = predict_labels(trainer, test_df["func"])

        for dataset in DATASETS:
            y_true = targets.loc[targets["dataset"] == dataset, "cwe_label"]
            y_pred = np.array(classes)[trainer.predict(target_data[dataset]).predictions.argmax(axis=1)]
            per_class_f1(y_true, y_pred).to_csv(RQ3_DIR / f"{dataset}_long_baseline_{name}_per_class.csv", index=False)
        history.to_csv(LONG_DIR / f"{name}_history.csv", index=False)
        per_class_f1(test_df["cwe_label"], test_pred).to_csv(LONG_DIR / f"{name}_per_class.csv", index=False)
        best_epoch = int(history.loc[history["eval_macro_f1"].idxmax(), "epoch"])
        print(name, "epochs:", epochs, "| best epoch:", best_epoch, "| test macro-F1:", round(macro_f1(test_df["cwe_label"], test_pred), 4))

### Control results

This cell compares three models on DiverseVul test, BigVul and MegaVul: the baseline (`baseline`), the long baseline (`long_baseline`) and SSL (`ssl`). For each dataset and label size, it shows the mean and the standard deviation over the 5 seeds of the macro-F1, the mean F1 of the rare classes, and the mean F1 of the common classes. It also shows two gains:

- `long_gain` = long baseline minus baseline: the effect of longer training alone.
- `ssl_vs_long` = SSL minus long baseline: the effect of the unlabeled data when both models make the same number of training steps. If this gain is close to 0, the SSL gain comes from longer training, not from the pseudo-labels.

The table is saved in `results/long_baseline_summary.csv`. The cell also counts in how many of the 5 runs SSL is better than the long baseline.

In [ ]:
MODELS = {"baseline": BASELINE_DIR, "long_baseline": LONG_DIR, "ssl": SSL_DIR}


def run_file(dataset, model_type, name):
    if dataset == "diversevul":
        return MODELS[model_type] / f"{name}_per_class.csv"
    return RQ3_DIR / f"{dataset}_{model_type}_{name}_per_class.csv"


control_runs = pd.concat(
    pd.read_csv(run_file(dataset, model_type, f"frac{frac}_seed{seed}")).assign(dataset=dataset, model=model_type, label_frac=frac, seed=seed)
    for dataset in ["diversevul"] + DATASETS
    for model_type in MODELS
    for frac in LABEL_FRACS
    for seed in SEEDS
)
keys = ["dataset", "label_frac", "seed", "model"]
macro = control_runs.groupby(keys)["f1"].mean().rename("macro_f1")
groups = control_runs.groupby(keys + ["group"])["f1"].mean().unstack("group")
control_per_run = pd.concat([macro, groups], axis=1).unstack("model")
for metric in ["macro_f1", "rare", "common"]:
    control_per_run[(metric, "long_gain")] = control_per_run[(metric, "long_baseline")] - control_per_run[(metric, "baseline")]
    control_per_run[(metric, "ssl_vs_long")] = control_per_run[(metric, "ssl")] - control_per_run[(metric, "long_baseline")]
control_summary = control_per_run.sort_index(axis=1).groupby(["dataset", "label_frac"]).agg(["mean", "std"]).round(3)
control_summary.to_csv(RESULTS_DIR / "long_baseline_summary.csv")

for metric in ["macro_f1", "rare", "common"]:
    print(metric)
    print(control_summary[metric].to_string())
    print()

ssl_wins = (control_per_run[("macro_f1", "ssl_vs_long")] > 0).groupby(["dataset", "label_frac"]).sum()
print("Runs (of 5) where SSL beats the long baseline on macro-F1:")
print(ssl_wins.to_string())

### Main RQ2 plot: SSL versus the long baseline

After the control runs finish, this cell plots the mean per-class gain of SSL over the long baseline against the full training-class counts. It uses the saved per-class results, so no model is trained again. The earlier SSL-versus-standard plot remains an extra comparison. The main plot is saved in `figures/ssl_gain_vs_long_per_class.pdf`, and the paired-seed mean gains are saved in `results/ssl_gain_vs_long_per_class.csv`.

On the GPU machine, also upload `src/plot_rq2_control.py` and keep it at that path under the project root. This cell reads that file.


In [ ]:
import runpy

plot_control = runpy.run_path(str(ROOT / "src" / "plot_rq2_control.py"))["make_plot"]
control_per_class_gain = plot_control(RESULTS_DIR, FIGURES_DIR)
print(control_per_class_gain.groupby("label_frac")["gain"].mean().to_string())
